# sweep-config-dict — ex2: validate a wandb sweep config dict and report all schema errors

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `sweep-config-dict`. Running the final beacon cell reports progress against the `Config: wandb sweep config dict` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: wandb sweep config dict` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sweep-config-dict`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sweep-config-dict"
DD_SUBTOPIC = "Config: wandb sweep config dict"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Sweep config dict — schema validation

Ex1 BUILT a sweep config. The deepening move is to VALIDATE one — given an arbitrary user-supplied dict, return the list of schema errors. This is what `wandb.sweep(cfg)` does internally before posting to the server.

**Required keys:** `'method'` (one of `'grid'`, `'random'`, `'bayes'`) and `'parameters'` (a non-empty dict-of-dicts).

**Conditional requirement:** if `method == 'bayes'`, `'metric'` MUST be present and contain `'name'` (str) + `'goal'` (`'minimize'` | `'maximize'`).

**Per-parameter shape:** every value in `parameters` must be a dict with EXACTLY ONE of: `'value'`, `'values'`, or `'distribution'` keys. More than one specifier is the typo trap (`{'value': 'adam', 'values': ['sgd']}`).

The validator returns a `list[str]` — empty means valid. Listing errors instead of raising on the first one lets the user fix all problems at once.

### Exercise 2 — validate a wandb sweep config dict and report all schema errors

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze an arbitrary user-supplied dict against the wandb sweep schema and return the full list of error messages — including missing required keys, bad method values, missing bayes metric, and per-parameter specifier mistakes.
> Keywords: wandb, sweep, validation, schema
> ```

**KCs targeted:** `sweep-config-top-level-schema`, `per-parameter-specifier-exactly-one`

Implement `ex2_validate_sweep_config(cfg)`. Return a `list[str]` of schema error messages. Empty list = valid.

Checks (in this order, but collect ALL failures, do not short-circuit):

1. `'method'` key present AND in `{'grid', 'random', 'bayes'}`. If missing: append `'missing required key: method'`. If wrong value: append `'method must be one of grid/random/bayes, got <value>'`.
2. If `method == 'bayes'`:
   - `'metric'` key must be present. If missing: append `'bayes method requires metric block'`.
   - If present, `metric` must be a dict containing `'name'` (str) AND `'goal'` in `{'minimize', 'maximize'}`. Append `'metric missing name'` and/or `'metric goal must be minimize or maximize'` as applicable.
3. `'parameters'` key present AND a non-empty dict. Missing → `'missing required key: parameters'`; empty → `'parameters dict is empty'`.
4. For each `(pname, pspec)` in `parameters`:
   - `pspec` must be a dict. If not → append `'parameters.<pname>: spec must be a dict'`.
   - Count how many of `'value'`, `'values'`, `'distribution'` keys appear. Must be EXACTLY 1. If 0 → append `'parameters.<pname>: needs one of value/values/distribution'`. If >1 → append `'parameters.<pname>: only one of value/values/distribution allowed'`.

Input: `cfg` — arbitrary dict.
Output: `list[str]` of error messages (any order acceptable; tests check `set` equality).

In [ ]:
def ex2_validate_sweep_config(cfg):
    errs = []
    # method
    if 'method' not in cfg:
        errs.append('missing required key: method')
    else:
        if cfg['method'] not in {'grid', 'random', 'bayes'}:
            errs.append(
                f"method must be one of grid/random/bayes, got {cfg['method']!r}"
            )
    # bayes → metric
    if cfg.get('method') == 'bayes':
        if 'metric' not in cfg:
            errs.append('bayes method requires metric block')
        else:
            m = cfg['metric']
            if not isinstance(m, dict) or 'name' not in m or not isinstance(m.get('name'), str):
                errs.append('metric missing name')
            if not isinstance(m, dict) or m.get('goal') not in {'minimize', 'maximize'}:
                errs.append('metric goal must be minimize or maximize')
    # parameters
    if 'parameters' not in cfg:
        errs.append('missing required key: parameters')
    else:
        p = cfg['parameters']
        if not isinstance(p, dict):
            errs.append('parameters must be a dict')
        elif len(p) == 0:
            errs.append('parameters dict is empty')
        else:
            for pname, pspec in p.items():
                if not isinstance(pspec, dict):
                    errs.append(f'parameters.{pname}: spec must be a dict')
                    continue
                n = sum(k in pspec for k in ('value', 'values', 'distribution'))
                if n == 0:
                    errs.append(f'parameters.{pname}: needs one of value/values/distribution')
                elif n > 1:
                    errs.append(f'parameters.{pname}: only one of value/values/distribution allowed')
    return errs


<details><summary>Solution</summary>

```python
def ex2_validate_sweep_config(cfg):
    errs = []
    # method
    if 'method' not in cfg:
        errs.append('missing required key: method')
    else:
        if cfg['method'] not in {'grid', 'random', 'bayes'}:
            errs.append(
                f"method must be one of grid/random/bayes, got {cfg['method']!r}"
            )
    # bayes → metric
    if cfg.get('method') == 'bayes':
        if 'metric' not in cfg:
            errs.append('bayes method requires metric block')
        else:
            m = cfg['metric']
            if not isinstance(m, dict) or 'name' not in m or not isinstance(m.get('name'), str):
                errs.append('metric missing name')
            if not isinstance(m, dict) or m.get('goal') not in {'minimize', 'maximize'}:
                errs.append('metric goal must be minimize or maximize')
    # parameters
    if 'parameters' not in cfg:
        errs.append('missing required key: parameters')
    else:
        p = cfg['parameters']
        if not isinstance(p, dict):
            errs.append('parameters must be a dict')
        elif len(p) == 0:
            errs.append('parameters dict is empty')
        else:
            for pname, pspec in p.items():
                if not isinstance(pspec, dict):
                    errs.append(f'parameters.{pname}: spec must be a dict')
                    continue
                n = sum(k in pspec for k in ('value', 'values', 'distribution'))
                if n == 0:
                    errs.append(f'parameters.{pname}: needs one of value/values/distribution')
                elif n > 1:
                    errs.append(f'parameters.{pname}: only one of value/values/distribution allowed')
    return errs
```

**Collect all errors, never short-circuit.** Reporting one error at a time forces the user into N round-trips for N typos. A list of errors lets them fix everything in one pass — the same UX as a compiler's diagnostics buffer.

**`sum(k in pspec for k in (...))` is the exactly-one count.** Booleans add as 0/1. Three keys → counts 0..3. The check `n == 1` is the canonical 'mutually exclusive' validator.

**Why `pspec.get('goal') not in {...}` even after `isinstance`.** Defensive against `metric={'goal': None}` — `None not in {'minimize', 'maximize'}` is True, so the error fires. Without the set check, `None == 'minimize'` would be False but no error would be raised.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()